<div dir="rtl" style="text-align:right">

# תרגול 12 — לראות איך הרשת עובדת

## CNN, Autoencoder, RNN ו־LSTM 


</div>

<div dir="rtl" style="text-align:right">

## מה עושים היום

1. לראות תמונה כטנזור.
2. לחשב Convolution ידנית.
3. להבין Padding, Stride ו־Pooling ויזואלית.
4. לעקוב אחרי Shapes בארכיטקטורת CNN.
5. להציג Feature Maps.
6. להבין Encoder, Bottleneck ו־Decoder.
7. להציג Reconstruction ו־Latent Space.
8. להפוך סדרת זמן לחלונות.
9. להבין RNN Unrolling ו־LSTM Gates.
10. להשוות LSTM ל־Baselines.

</div>

<div dir="rtl" style="text-align:right">

# חלק א׳ — תמונות, Convolution ו־CNN

## 0. Setup

</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)
torch.set_num_threads(1)

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

In [ ]:
def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def show_matrix_with_values(matrix, title, fmt=".1f", cmap="gray"):
    values = to_numpy(matrix)

    plt.figure(figsize=(6, 6))
    plt.imshow(values, cmap=cmap)

    for row in range(values.shape[0]):
        for col in range(values.shape[1]):
            plt.text(
                col,
                row,
                format(values[row, col], fmt),
                ha="center",
                va="center"
            )

    plt.title(title)
    plt.colorbar()
    plt.show()


def draw_horizontal_architecture(blocks, title):
    width = max(10, len(blocks) * 2.0)

    fig = plt.figure(figsize=(width, 3.5))
    ax = fig.add_axes([0.02, 0.10, 0.96, 0.80])

    box_width = 1.35
    box_height = 1.15
    gap = 0.65

    for index, block in enumerate(blocks):
        name, shape = block
        x = index * (box_width + gap)

        rectangle = plt.Rectangle(
            (x, 0.65),
            box_width,
            box_height,
            fill=False,
            linewidth=2
        )
        ax.add_patch(rectangle)

        ax.text(
            x + box_width / 2,
            1.25,
            name,
            ha="center",
            va="center",
            fontsize=10
        )

        ax.text(
            x + box_width / 2,
            0.35,
            shape,
            ha="center",
            va="center",
            fontsize=9
        )

        if index < len(blocks) - 1:
            ax.annotate(
                "",
                xy=(x + box_width + gap * 0.82, 1.20),
                xytext=(x + box_width + gap * 0.18, 1.20),
                arrowprops={"arrowstyle": "->", "linewidth": 1.5}
            )

    total_width = len(blocks) * box_width + (len(blocks) - 1) * gap
    ax.set_xlim(-0.2, total_width + 0.2)
    ax.set_ylim(0, 2.4)
    ax.set_title(title)
    ax.axis("off")
    plt.show()

<div dir="rtl" style="text-align:right">

## 1. טעינת תמונות ספרות

נשתמש ב־`load_digits`: תמונות grayscale בגודל 8×8.

</div>

In [ ]:
digits = load_digits()

images = digits.images.astype(np.float32) / 16.0
labels = digits.target.astype(np.int64)

X_train_np, X_temp_np, y_train_np, y_temp_np = train_test_split(
    images,
    labels,
    test_size=0.30,
    random_state=42,
    stratify=labels
)

X_val_np, X_test_np, y_val_np, y_test_np = train_test_split(
    X_temp_np,
    y_temp_np,
    test_size=0.50,
    random_state=42,
    stratify=y_temp_np
)

X_train_img = torch.tensor(X_train_np, dtype=torch.float32).unsqueeze(1)
X_val_img = torch.tensor(X_val_np, dtype=torch.float32).unsqueeze(1)
X_test_img = torch.tensor(X_test_np, dtype=torch.float32).unsqueeze(1)

y_train_digits = torch.tensor(y_train_np, dtype=torch.long)
y_val_digits = torch.tensor(y_val_np, dtype=torch.long)
y_test_digits = torch.tensor(y_test_np, dtype=torch.long)

train_loader_digits = DataLoader(
    TensorDataset(X_train_img, y_train_digits),
    batch_size=64,
    shuffle=True
)

val_loader_digits = DataLoader(
    TensorDataset(X_val_img, y_val_digits),
    batch_size=128,
    shuffle=False
)

test_loader_digits = DataLoader(
    TensorDataset(X_test_img, y_test_digits),
    batch_size=128,
    shuffle=False
)

print("All images:", images.shape)
print("Train:", X_train_img.shape, y_train_digits.shape)
print("Validation:", X_val_img.shape, y_val_digits.shape)
print("Test:", X_test_img.shape, y_test_digits.shape)

<div dir="rtl" style="text-align:right">

## תרגיל 1 — תמונה כמטריצה וכטנזור

בחרו תמונה אחת והציגו:

1. את התמונה.
2. Heatmap עם ערכי הפיקסלים.
3. Shape לאחר הוספת Batch ו־Channel.

</div>

In [ ]:
# TODO:
# image = ...
# label = ...
# הציגו תמונה
# השתמשו ב-show_matrix_with_values
# צרו Tensor בצורה [1, 1, 8, 8]

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
image = images[0]
label = labels[0]

plt.figure(figsize=(4, 4))
plt.imshow(image, cmap="gray")
plt.title(f"Digit label: {label}")
plt.axis("off")
plt.show()

show_matrix_with_values(
    image,
    "Image as Pixel Values",
    fmt=".1f"
)

image_tensor = torch.tensor(
    image,
    dtype=torch.float32
).unsqueeze(0).unsqueeze(0)

print("Image tensor shape:", image_tensor.shape)
```

</details>

<div dir="rtl" style="text-align:right">

## Channels

הריצו את התא הבא כדי לראות תמונת RGB ושלושת ה־Channels.

</div>

In [ ]:
axis_values = np.linspace(0, 1, 8, dtype=np.float32)

red_channel = np.tile(axis_values, (8, 1))
green_channel = red_channel.T
blue_channel = np.flipud(red_channel)

rgb_image = np.stack(
    [red_channel, green_channel, blue_channel],
    axis=-1
)

plt.figure(figsize=(5, 5))
plt.imshow(rgb_image)
plt.title("Synthetic RGB Image")
plt.axis("off")
plt.show()

plt.figure(figsize=(5, 5))
plt.imshow(red_channel, cmap="gray")
plt.title("Red Channel")
plt.colorbar()
plt.show()

plt.figure(figsize=(5, 5))
plt.imshow(green_channel, cmap="gray")
plt.title("Green Channel")
plt.colorbar()
plt.show()

plt.figure(figsize=(5, 5))
plt.imshow(blue_channel, cmap="gray")
plt.title("Blue Channel")
plt.colorbar()
plt.show()

print("NumPy RGB shape [H, W, C]:", rgb_image.shape)

rgb_tensor = torch.tensor(rgb_image).permute(2, 0, 1)
print("PyTorch image shape [C, H, W]:", rgb_tensor.shape)

<div dir="rtl" style="text-align:right">

## 2. Convolution ידנית

נשתמש ב־Kernel לזיהוי קצה אנכי.

</div>

<div dir="rtl" style="text-align:right">

## תרגיל 2 — Patch × Kernel

1. בחרו Patch בגודל 3×3.
2. הכפילו אותו ב־Kernel.
3. סכמו.
4. הפעילו `F.conv2d`.
5. ודאו שהערך הידני שווה לערך המתאים ב־Feature Map.

</div>

In [ ]:
# TODO:
# sample_image = ...
# vertical_kernel = ...
# patch = ...
# elementwise_product = ...
# manual_output = ...
# F.conv2d(...)

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
sample_image = X_train_img[0:1]
sample_label = y_train_digits[0].item()

vertical_kernel = torch.tensor(
    [
        [-1.0, 0.0, 1.0],
        [-1.0, 0.0, 1.0],
        [-1.0, 0.0, 1.0]
    ],
    dtype=torch.float32
)

patch = sample_image[0, 0, 2:5, 2:5]
elementwise_product = patch * vertical_kernel
manual_output = elementwise_product.sum()

show_matrix_with_values(
    sample_image[0, 0],
    f"Input Digit: {sample_label}",
    fmt=".1f"
)

show_matrix_with_values(
    vertical_kernel,
    "Vertical Edge Kernel",
    fmt=".0f",
    cmap=None
)

show_matrix_with_values(
    patch,
    "Selected 3×3 Image Patch",
    fmt=".2f"
)

show_matrix_with_values(
    elementwise_product,
    "Patch × Kernel",
    fmt=".2f",
    cmap=None
)

print("Manual convolution output:", manual_output.item())

kernel_tensor = vertical_kernel.view(1, 1, 3, 3)

vertical_feature_map = F.conv2d(
    sample_image,
    kernel_tensor,
    stride=1,
    padding=0
)

plt.figure(figsize=(5, 5))
plt.imshow(vertical_feature_map[0, 0].numpy())
plt.title("Vertical Edge Feature Map")
plt.colorbar()
plt.show()

print(
    "conv2d value at output [2, 2]:",
    vertical_feature_map[0, 0, 2, 2].item()
)
```

</details>

<div dir="rtl" style="text-align:right">

## תרגיל 3 — Kernels שונים

הפעילו Vertical Edge, Horizontal Edge, Blur ו־Sharpen.  
הציגו כל Kernel וכל Output בגרף נפרד.

</div>

In [ ]:
# TODO:
# הגדירו kernels והפעילו F.conv2d

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
kernels = {
    "Vertical edge": torch.tensor(
        [
            [-1.0, 0.0, 1.0],
            [-1.0, 0.0, 1.0],
            [-1.0, 0.0, 1.0]
        ]
    ),
    "Horizontal edge": torch.tensor(
        [
            [-1.0, -1.0, -1.0],
            [0.0, 0.0, 0.0],
            [1.0, 1.0, 1.0]
        ]
    ),
    "Blur": torch.ones((3, 3)) / 9.0,
    "Sharpen": torch.tensor(
        [
            [0.0, -1.0, 0.0],
            [-1.0, 5.0, -1.0],
            [0.0, -1.0, 0.0]
        ]
    )
}

for kernel_name, kernel in kernels.items():
    output = F.conv2d(
        sample_image,
        kernel.float().view(1, 1, 3, 3),
        padding=1
    )

    show_matrix_with_values(
        kernel,
        f"{kernel_name} Kernel",
        fmt=".2f",
        cmap=None
    )

    plt.figure(figsize=(5, 5))
    plt.imshow(output[0, 0].numpy())
    plt.title(f"{kernel_name} Output")
    plt.colorbar()
    plt.show()
```

</details>

<div dir="rtl" style="text-align:right">

## 3. Padding ו־Stride

</div>

<div dir="rtl" style="text-align:right">

## תרגיל 4 — השפעת Padding ו־Stride

1. הציגו Zero Padding.
2. הריצו ארבעה Configurations.
3. הדפיסו Shape.
4. הציגו כל Feature Map בנפרד.

</div>

In [ ]:
# TODO:
# padded image
# configurations
# outputs and plots

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
padded_image = F.pad(
    sample_image,
    pad=(1, 1, 1, 1),
    mode="constant",
    value=0
)

show_matrix_with_values(
    padded_image[0, 0],
    "Input Image with Zero Padding = 1",
    fmt=".1f"
)

configurations = [
    {"padding": 0, "stride": 1},
    {"padding": 1, "stride": 1},
    {"padding": 0, "stride": 2},
    {"padding": 1, "stride": 2}
]

results = []

for config in configurations:
    output = F.conv2d(
        sample_image,
        kernel_tensor,
        padding=config["padding"],
        stride=config["stride"]
    )

    results.append({
        "padding": config["padding"],
        "stride": config["stride"],
        "height": output.shape[-2],
        "width": output.shape[-1]
    })

    print(
        config,
        "→",
        tuple(output.shape)
    )

    plt.figure(figsize=(5, 5))
    plt.imshow(output[0, 0].numpy())
    plt.title(
        f"padding={config['padding']}, "
        f"stride={config['stride']}"
    )
    plt.colorbar()
    plt.show()

pd.DataFrame(results)
```

</details>

<div dir="rtl" style="text-align:right">

## 4. Pooling

</div>

<div dir="rtl" style="text-align:right">

## תרגיל 5 — Max Pooling ו־Average Pooling

הציגו Input, Max Pooling Output ו־Average Pooling Output.

</div>

In [ ]:
# TODO:
# pool_input
# max_pool2d
# avg_pool2d
# visualizations

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
pool_input = torch.tensor(
    [
        [1.0, 2.0, 5.0, 0.0],
        [3.0, 4.0, 1.0, 2.0],
        [0.0, 1.0, 7.0, 6.0],
        [2.0, 3.0, 5.0, 8.0]
    ]
)

pool_tensor = pool_input.view(1, 1, 4, 4)

max_pooled = F.max_pool2d(
    pool_tensor,
    kernel_size=2,
    stride=2
)

avg_pooled = F.avg_pool2d(
    pool_tensor,
    kernel_size=2,
    stride=2
)

show_matrix_with_values(
    pool_input,
    "Input Feature Map for Pooling",
    fmt=".0f",
    cmap=None
)

show_matrix_with_values(
    max_pooled[0, 0],
    "Max Pooling Output",
    fmt=".1f",
    cmap=None
)

show_matrix_with_values(
    avg_pooled[0, 0],
    "Average Pooling Output",
    fmt=".2f",
    cmap=None
)
```

</details>

<div dir="rtl" style="text-align:right">

## 5. ארכיטקטורת CNN

</div>

In [ ]:
cnn_blocks = [
    ("Input", "1×8×8"),
    ("Conv + ReLU", "8×8×8"),
    ("MaxPool", "8×4×4"),
    ("Conv + ReLU", "16×4×4"),
    ("MaxPool", "16×2×2"),
    ("Flatten", "64"),
    ("Linear", "10 logits")
]

draw_horizontal_architecture(
    cnn_blocks,
    "TinyCNN Architecture and Tensor Shapes"
)

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.classifier = nn.Linear(16 * 2 * 2, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)


def trace_cnn_shapes(model, x):
    print("Input:", tuple(x.shape))
    x = model.conv1(x)
    print("After conv1:", tuple(x.shape))
    x = F.relu(x)
    x = model.pool(x)
    print("After pool1:", tuple(x.shape))
    x = model.conv2(x)
    print("After conv2:", tuple(x.shape))
    x = F.relu(x)
    x = model.pool(x)
    print("After pool2:", tuple(x.shape))
    x = torch.flatten(x, start_dim=1)
    print("After flatten:", tuple(x.shape))
    x = model.classifier(x)
    print("Output:", tuple(x.shape))


def evaluate_classifier(model, loader):
    model.eval()
    loss_fn = nn.CrossEntropyLoss()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for batch_images, batch_labels in loader:
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.to(device)

            logits = model(batch_images)
            loss = loss_fn(logits, batch_labels)

            total_loss += loss.item() * len(batch_images)
            total_correct += (
                logits.argmax(dim=1) == batch_labels
            ).sum().item()
            total_examples += len(batch_images)

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples
    }


def train_cnn(model, train_loader, val_loader, epochs=12):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(epochs):
        model.train()

        total_loss = 0.0
        total_examples = 0

        for batch_images, batch_labels in train_loader:
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.to(device)

            optimizer.zero_grad()
            logits = model(batch_images)
            loss = loss_fn(logits, batch_labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(batch_images)
            total_examples += len(batch_images)

        val_metrics = evaluate_classifier(model, val_loader)

        history["train_loss"].append(total_loss / total_examples)
        history["val_loss"].append(val_metrics["loss"])
        history["val_accuracy"].append(val_metrics["accuracy"])

    return pd.DataFrame(history)

<div dir="rtl" style="text-align:right">

## תרגיל 6 — אימון CNN

צרו את המודל, עקבו אחרי Shapes, אמנו 12 Epochs והציגו Loss ו־Accuracy.

</div>

In [ ]:
# TODO:
# create model
# trace shapes
# train
# evaluate
# plots

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
torch.manual_seed(42)

cnn_model = TinyCNN().to(device)

trace_cnn_shapes(
    cnn_model,
    X_train_img[:4].to(device)
)

cnn_history = train_cnn(
    cnn_model,
    train_loader_digits,
    val_loader_digits,
    epochs=12
)

test_metrics = evaluate_classifier(
    cnn_model,
    test_loader_digits
)

print(test_metrics)

plt.figure(figsize=(8, 5))
plt.plot(cnn_history["train_loss"], label="train")
plt.plot(cnn_history["val_loss"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("CNN Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(cnn_history["val_accuracy"])
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("CNN Validation Accuracy")
plt.show()
```

</details>

<div dir="rtl" style="text-align:right">

## תרגיל 7 — Feature Maps

הציגו תמונה מקורית, ארבע מפות מ־Conv1 וארבע מפות מ־Conv2.

</div>

In [ ]:
# TODO:
# forward to conv1 and conv2
# separate plots

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
cnn_model.eval()

feature_sample = X_test_img[0:1].to(device)

with torch.no_grad():
    conv1_output = F.relu(cnn_model.conv1(feature_sample))
    pool1_output = cnn_model.pool(conv1_output)
    conv2_output = F.relu(cnn_model.conv2(pool1_output))

plt.figure(figsize=(4, 4))
plt.imshow(feature_sample[0, 0].cpu().numpy(), cmap="gray")
plt.title("Original Test Image")
plt.axis("off")
plt.show()

for channel in range(4):
    plt.figure(figsize=(4, 4))
    plt.imshow(conv1_output[0, channel].cpu().numpy())
    plt.title(f"Conv1 Feature Map {channel}")
    plt.colorbar()
    plt.show()

for channel in range(4):
    plt.figure(figsize=(4, 4))
    plt.imshow(conv2_output[0, channel].cpu().numpy())
    plt.title(f"Conv2 Feature Map {channel}")
    plt.colorbar()
    plt.show()
```

</details>

<div dir="rtl" style="text-align:right">

## 6. Autoencoder

</div>

In [ ]:
autoencoder_blocks = [
    ("Input", "64 pixels"),
    ("Encoder", "64 → 32"),
    ("Bottleneck", "2 values"),
    ("Decoder", "2 → 32"),
    ("Output", "64 pixels")
]

draw_horizontal_architecture(
    autoencoder_blocks,
    "Autoencoder Architecture"
)

In [ ]:
class DigitAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

        self.decoder = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.Sigmoid()
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        return reconstruction, latent


def train_autoencoder(model, train_loader, val_loader, epochs=30):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_examples = 0

        for batch_images, _ in train_loader:
            flat = batch_images.to(device).view(len(batch_images), -1)

            optimizer.zero_grad()
            reconstruction, latent = model(flat)
            loss = loss_fn(reconstruction, flat)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(batch_images)
            total_examples += len(batch_images)

        model.eval()
        val_total = 0.0
        val_examples = 0

        with torch.no_grad():
            for batch_images, _ in val_loader:
                flat = batch_images.to(device).view(len(batch_images), -1)
                reconstruction, latent = model(flat)
                loss = loss_fn(reconstruction, flat)

                val_total += loss.item() * len(batch_images)
                val_examples += len(batch_images)

        history["train_loss"].append(total_loss / total_examples)
        history["val_loss"].append(val_total / val_examples)

    return pd.DataFrame(history)

<div dir="rtl" style="text-align:right">

## תרגיל 8 — אימון Autoencoder

</div>

In [ ]:
# TODO:
# create autoencoder
# train
# plot reconstruction loss

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
torch.manual_seed(42)

autoencoder = DigitAutoencoder().to(device)

autoencoder_history = train_autoencoder(
    autoencoder,
    train_loader_digits,
    val_loader_digits,
    epochs=30
)

plt.figure(figsize=(8, 5))
plt.plot(autoencoder_history["train_loss"], label="train")
plt.plot(autoencoder_history["val_loss"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Reconstruction MSE")
plt.title("Autoencoder Reconstruction Loss")
plt.legend()
plt.show()
```

</details>

<div dir="rtl" style="text-align:right">

## תרגיל 9 — Reconstruction ו־Latent Space

הציגו Original, Reconstruction, Error Map ו־Latent Space דו־ממדי.

</div>

In [ ]:
# TODO:
# reconstruct image
# error map
# encode test set
# latent scatter

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
autoencoder.eval()

with torch.no_grad():
    flat_sample = X_test_img[0:1].to(device).view(1, -1)
    reconstructed, latent = autoencoder(flat_sample)

original_image = flat_sample[0].cpu().numpy().reshape(8, 8)
reconstructed_image = reconstructed[0].cpu().numpy().reshape(8, 8)
error_map = np.abs(original_image - reconstructed_image)

plt.figure(figsize=(4, 4))
plt.imshow(original_image, cmap="gray")
plt.title("Original Image")
plt.axis("off")
plt.show()

plt.figure(figsize=(4, 4))
plt.imshow(reconstructed_image, cmap="gray")
plt.title("Reconstruction")
plt.axis("off")
plt.show()

plt.figure(figsize=(4, 4))
plt.imshow(error_map)
plt.title("Absolute Reconstruction Error")
plt.colorbar()
plt.show()

latent_vectors = []
latent_labels = []

with torch.no_grad():
    for batch_images, batch_labels in test_loader_digits:
        flat = batch_images.to(device).view(len(batch_images), -1)
        latent = autoencoder.encoder(flat)

        latent_vectors.append(latent.cpu().numpy())
        latent_labels.append(batch_labels.numpy())

latent_vectors = np.concatenate(latent_vectors)
latent_labels = np.concatenate(latent_labels)

plt.figure(figsize=(8, 6))
plt.scatter(
    latent_vectors[:, 0],
    latent_vectors[:, 1],
    c=latent_labels,
    alpha=0.7
)
plt.xlabel("Latent dimension 1")
plt.ylabel("Latent dimension 2")
plt.title("2D Autoencoder Latent Space")
plt.colorbar()
plt.show()
```

</details>

<div dir="rtl" style="text-align:right">

# חלק ב׳ — סדרות זמן, RNN ו־LSTM

## 7. יצירת סדרת זמן וחלונות

</div>

In [ ]:
rng = np.random.default_rng(42)

time = np.arange(400)

series = (
    0.70 * np.sin(0.08 * time)
    + 0.30 * np.sin(0.19 * time)
    + 0.0015 * time
    + rng.normal(0, 0.04, size=len(time))
).astype(np.float32)

split_index = 300

plt.figure(figsize=(10, 4))
plt.plot(time, series)
plt.axvline(split_index, linestyle="--")
plt.xlabel("Time")
plt.ylabel("Value")
plt.title("Synthetic Time Series and Train/Test Split")
plt.show()

train_mean = series[:split_index].mean()
train_std = series[:split_index].std()

series_scaled = (series - train_mean) / train_std
lookback = 20


def make_windows(values, lookback):
    X_windows = []
    y_targets = []

    for index in range(len(values) - lookback):
        X_windows.append(values[index:index + lookback])
        y_targets.append(values[index + lookback])

    return (
        np.asarray(X_windows, dtype=np.float32),
        np.asarray(y_targets, dtype=np.float32)
    )


X_train_ts_np, y_train_ts_np = make_windows(
    series_scaled[:split_index],
    lookback
)

test_context = series_scaled[split_index - lookback:]

X_test_ts_np, y_test_ts_np = make_windows(
    test_context,
    lookback
)

X_train_ts = torch.tensor(X_train_ts_np).unsqueeze(-1)
y_train_ts = torch.tensor(y_train_ts_np).unsqueeze(-1)
X_test_ts = torch.tensor(X_test_ts_np).unsqueeze(-1)
y_test_ts = torch.tensor(y_test_ts_np).unsqueeze(-1)

train_loader_ts = DataLoader(
    TensorDataset(X_train_ts, y_train_ts),
    batch_size=32,
    shuffle=True
)

print("Train windows:", X_train_ts.shape, y_train_ts.shape)
print("Test windows:", X_test_ts.shape, y_test_ts.shape)

<div dir="rtl" style="text-align:right">

## תרגיל 10 — Sliding Window

</div>

In [ ]:
# TODO:
# plot one input window and next target

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
window_index = 40

window = X_train_ts_np[window_index]
target = y_train_ts_np[window_index]

plt.figure(figsize=(8, 4))
plt.plot(
    range(lookback),
    window,
    marker="o",
    label="input window"
)
plt.scatter(
    [lookback],
    [target],
    marker="x",
    s=100,
    label="next-step target"
)
plt.xlabel("Position in window")
plt.ylabel("Scaled value")
plt.title("Lookback Window and Next-Step Target")
plt.legend()
plt.show()
```

</details>

<div dir="rtl" style="text-align:right">

## 8. RNN Unrolling

</div>

In [ ]:
def draw_rnn_unrolling(sequence_length=5):
    fig = plt.figure(figsize=(11, 4))
    ax = fig.add_axes([0.03, 0.08, 0.94, 0.84])

    for step in range(sequence_length):
        x = step * 1.8

        rectangle = plt.Rectangle(
            (x, 1.15),
            1.15,
            0.85,
            fill=False,
            linewidth=2
        )
        ax.add_patch(rectangle)

        ax.text(x + 0.575, 1.58, "RNN Cell", ha="center", va="center")

        ax.annotate(
            "",
            xy=(x + 0.575, 1.15),
            xytext=(x + 0.575, 0.38),
            arrowprops={"arrowstyle": "->"}
        )

        ax.text(x + 0.575, 0.18, f"x{step + 1}", ha="center")

        ax.annotate(
            "",
            xy=(x + 0.575, 2.55),
            xytext=(x + 0.575, 2.00),
            arrowprops={"arrowstyle": "->"}
        )

        ax.text(x + 0.575, 2.72, f"h{step + 1}", ha="center")

        if step < sequence_length - 1:
            ax.annotate(
                "",
                xy=(x + 1.75, 1.58),
                xytext=(x + 1.15, 1.58),
                arrowprops={"arrowstyle": "->", "linewidth": 1.5}
            )

    ax.text(-0.65, 1.58, "h0", ha="center", va="center")
    ax.annotate(
        "",
        xy=(0, 1.58),
        xytext=(-0.45, 1.58),
        arrowprops={"arrowstyle": "->"}
    )

    ax.set_xlim(-0.9, sequence_length * 1.8)
    ax.set_ylim(0, 3.05)
    ax.set_title("RNN Unrolled Through Time — Same Weights at Every Step")
    ax.axis("off")
    plt.show()


draw_rnn_unrolling()

<div dir="rtl" style="text-align:right">

## 9. LSTM Gates

</div>

In [ ]:
def draw_lstm_gate_diagram():
    fig = plt.figure(figsize=(11, 5))
    ax = fig.add_axes([0.03, 0.07, 0.94, 0.86])

    gate_names = [
        ("Forget Gate", 0.8),
        ("Input Gate", 3.2),
        ("Candidate", 5.6),
        ("Output Gate", 8.0)
    ]

    for name, x in gate_names:
        rectangle = plt.Rectangle(
            (x, 1.35),
            1.55,
            0.85,
            fill=False,
            linewidth=2
        )
        ax.add_patch(rectangle)
        ax.text(x + 0.775, 1.78, name, ha="center", va="center")

    ax.text(0.1, 3.75, "cₜ₋₁", fontsize=13)
    ax.annotate(
        "",
        xy=(10.1, 3.75),
        xytext=(0.55, 3.75),
        arrowprops={"arrowstyle": "->", "linewidth": 2}
    )
    ax.text(10.2, 3.75, "cₜ", fontsize=13)
    ax.text(0.1, 0.50, "xₜ and hₜ₋₁", fontsize=12)

    for _, x in gate_names:
        ax.annotate(
            "",
            xy=(x + 0.775, 1.35),
            xytext=(1.05, 0.72),
            arrowprops={"arrowstyle": "->"}
        )

    ax.text(8.85, 3.02, "hₜ", fontsize=13)
    ax.set_xlim(0, 10.8)
    ax.set_ylim(0, 4.8)
    ax.set_title("LSTM Cell — Learn What to Forget, Add, and Expose")
    ax.axis("off")
    plt.show()


draw_lstm_gate_diagram()

<div dir="rtl" style="text-align:right">

## תרגיל 11 — Shapes של LSTM

</div>

In [ ]:
sample_batch = torch.randn(4, 20, 1)

shape_lstm = nn.LSTM(
    input_size=1,
    hidden_size=16,
    batch_first=True
)

output, (h_n, c_n) = shape_lstm(sample_batch)

print("Input:", sample_batch.shape)
print("Output:", output.shape)
print("h_n:", h_n.shape)
print("c_n:", c_n.shape)

draw_horizontal_architecture(
    [
        ("Input", "[B, T, 1]"),
        ("LSTM", "hidden=16"),
        ("All outputs", "[B, T, 16]"),
        ("Final hₙ", "[1, B, 16]"),
        ("Linear", "[B, 1]")
    ],
    "LSTM Forecasting Architecture and Tensor Shapes"
)

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, hidden_size=16):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, (h_n, c_n) = self.lstm(x)
        final_hidden = h_n[-1]
        return self.output_layer(final_hidden)


def train_forecaster(model, train_loader, epochs=100):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()
    losses = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_examples = 0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = loss_fn(predictions, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(batch_X)
            total_examples += len(batch_X)

        losses.append(total_loss / total_examples)

    return losses

<div dir="rtl" style="text-align:right">

## תרגיל 12 — אימון LSTM

</div>

In [ ]:
# TODO:
# create model
# train 100 epochs
# plot loss

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
torch.manual_seed(42)

lstm_model = LSTMForecaster(hidden_size=16).to(device)

lstm_losses = train_forecaster(
    lstm_model,
    train_loader_ts,
    epochs=100
)

plt.figure(figsize=(8, 5))
plt.plot(lstm_losses)
plt.xlabel("Epoch")
plt.ylabel("Train MSE")
plt.title("LSTM Forecasting Training Loss")
plt.show()
```

</details>

<div dir="rtl" style="text-align:right">

## תרגיל 13 — LSTM מול Baselines

השוו Last Value, Window Mean ו־LSTM באמצעות RMSE וגרפים נפרדים.

</div>

In [ ]:
# TODO:
# predictions
# inverse scaling
# baselines
# RMSE table
# plots

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
lstm_model.eval()

with torch.no_grad():
    predictions_scaled = (
        lstm_model(X_test_ts.to(device))
        .cpu()
        .numpy()
        .ravel()
    )

actual_values = y_test_ts_np * train_std + train_mean
lstm_predictions = predictions_scaled * train_std + train_mean
last_value_baseline = X_test_ts_np[:, -1] * train_std + train_mean
mean_window_baseline = X_test_ts_np.mean(axis=1) * train_std + train_mean


def rmse(actual, predicted):
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))


rmse_table = pd.DataFrame({
    "model": [
        "Last value baseline",
        "Window mean baseline",
        "LSTM"
    ],
    "RMSE": [
        rmse(actual_values, last_value_baseline),
        rmse(actual_values, mean_window_baseline),
        rmse(actual_values, lstm_predictions)
    ]
})

display(rmse_table)

plt.figure(figsize=(10, 4))
plt.plot(actual_values, label="actual")
plt.plot(mean_window_baseline, label="window mean baseline")
plt.xlabel("Test step")
plt.ylabel("Value")
plt.title("Actual vs Mean-Window Baseline")
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(actual_values, label="actual")
plt.plot(last_value_baseline, label="last-value baseline")
plt.xlabel("Test step")
plt.ylabel("Value")
plt.title("Actual vs Last-Value Baseline")
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(actual_values, label="actual")
plt.plot(lstm_predictions, label="LSTM prediction")
plt.xlabel("Test step")
plt.ylabel("Value")
plt.title("Actual vs LSTM Forecast")
plt.legend()
plt.show()
```

</details>